In [1]:
import cv2
import numpy as np
import math
import pyautogui as p

p.FAILSAFE = False
 
# Function to calculate the distance between two points
def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

# Set up webcam capture
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

# Create a window for color adjustments
cv2.namedWindow("Color Adjustments")
cv2.createTrackbar("HMin", "Color Adjustments", 0, 179, lambda x: None)
cv2.createTrackbar("HMax", "Color Adjustments", 20, 179, lambda x: None)
cv2.createTrackbar("SMin", "Color Adjustments", 48, 255, lambda x: None)
cv2.createTrackbar("SMax", "Color Adjustments", 255, 255, lambda x: None)
cv2.createTrackbar("VMin", "Color Adjustments", 80, 255, lambda x: None)
cv2.createTrackbar("VMax", "Color Adjustments", 255, 255, lambda x: None)

while True:
    ret, frame = cap.read()  # Capture frame from camera
    if not ret:  # Check if frame is captured successfully
        print("Failed to capture image")
        break  # Exit the loop if frame capture fails

    frame = cv2.flip(frame, 1)  # Flip horizontally for a mirror effect
    frame = cv2.resize(frame, (600, 500))
    cv2.rectangle(frame, (0, 1), (300, 500), (255, 0, 0), 0)
    crop_image = frame[1:500, 0:300]

    # Convert frame to HSV
    hsv = cv2.cvtColor(crop_image, cv2.COLOR_BGR2HSV)

    # Get HSV values from trackbars
    h_min = cv2.getTrackbarPos("HMin", "Color Adjustments")
    h_max = cv2.getTrackbarPos("HMax", "Color Adjustments")
    s_min = cv2.getTrackbarPos("SMin", "Color Adjustments")
    s_max = cv2.getTrackbarPos("SMax", "Color Adjustments")
    v_min = cv2.getTrackbarPos("VMin", "Color Adjustments")
    v_max = cv2.getTrackbarPos("VMax", "Color Adjustments")

    # Define the range of skin color in HSV
    lower_skin = np.array([h_min, s_min, v_min])
    upper_skin = np.array([h_max, s_max, v_max])

    # Create a mask
    mask = cv2.inRange(hsv, lower_skin, upper_skin)
    mask = cv2.GaussianBlur(mask, (5, 5), 100)

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Find the largest contour
        max_contour = max(contours, key=lambda x: cv2.contourArea(x))
        cv2.drawContours(crop_image, [max_contour], -1, (0, 255, 0), 2)

        # Find the convex hull
        hull = cv2.convexHull(max_contour)
        cv2.drawContours(crop_image, [hull], -1, (0, 0, 255), 2)

        # Find the convexity defects
        hull_indices = cv2.convexHull(max_contour, returnPoints=False)
        defects = cv2.convexityDefects(max_contour, hull_indices)

        count_defects = 0

        if defects is not None:
            for i in range(defects.shape[0]):
                s, e, f, d = defects[i, 0]
                start = tuple(max_contour[s][0])
                end = tuple(max_contour[e][0])
                far = tuple(max_contour[f][0])

                # Calculate distances
                a = distance(start, end)
                b = distance(start, far)
                c = distance(end, far)

                # Calculate angle
                if b != 0 and c != 0:  # Ensure b and c are not zero
                    cosine_angle = (b ** 2 + c ** 2 - a ** 2) / (2 * b * c)
                    angle = math.acos(cosine_angle) * 180 / math.pi

                    # Count the number of fingers
                    if angle <= 90:  # If the angle is less than or equal to 90 degrees
                        count_defects += 1
                        cv2.circle(crop_image, far, 8, (0, 0, 255), -1)

        # Add 1 to the count of fingers (the thumb is not counted in defects)
        total_fingers = count_defects + 1

        # Display the number of fingers on the frame
        cv2.putText(frame, f'Fingers: {total_fingers}', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # Perform actions based on the number of fingers
        if total_fingers > 0:  # Only perform actions if at least one finger is detected
            if total_fingers == 1:
                p.press('up')  # Volume up
            elif total_fingers == 2:
                p.press('down')  # Volume down
            elif total_fingers == 3:
                p.press('space')  # Play/pause
            elif total_fingers == 4:
                p.hotkey('ctrl', 'right')  # Next track
            elif total_fingers == 5:
                p.hotkey('ctrl', 'left')  # Previous track
    else:
        # Display zero fingers if no contours are found
        cv2.putText(frame, 'Fingers: 0', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Show the processed frame and mask
    cv2.imshow("Frame", frame)
    cv2.imshow("Mask", mask)

    # Exit the loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the webcam and destroy all OpenCV windows
cap.release()
cv2.destroyAllWindows()          

2025-01-11 18:30:53.475 Python[22884:868608] +[IMKClient subclass]: chose IMKClient_Modern
2025-01-11 18:30:53.475 Python[22884:868608] +[IMKInputSession subclass]: chose IMKInputSession_Modern
